# Metin ve Açıklama

Bu notebook, PDS Handbook (TR) web sayfasının **Türkçe Jupyter karşılığıdır** — aynı açıklamalar, ders notları ve kod örnekleri.

| | |
|---|---|
| **Web sayfası** | `chapters/04-matplotlib/09-text-and-annotation.html` |
| **Çalıştırma** | JupyterLab, VS Code veya Colab — hücreleri **yukarıdan aşağı** sırayla (`Shift+Enter`) |
| **Bağımlılık** | Kod hücreleri birbirine bağlıdır; hata alırsanız önce üsttekileri çalıştırın |

> **Kaynak:** Jake VanderPlas, *Python Data Science Handbook* — Türkçe ders uyarlaması



Orijinal: 04.09 Text and Annotation

İyi bir görselleştirme okuyucuyu yönlendirerek figürün bir hikâye anlatmasını sağlar.
    Bazen hikâye tamamen görsel olabilir; bazen küçük metin ipuçları ve etiketler gerekir.
    En temel açıklamalar eksen etiketleri ve başlıklardır; seçenekler bununla sınırlı değildir.
    Veriye bakalım ve ilginç bilgiyi iletmek için nasıl görselleştirip açıklayabileceğimize bakalım.
    Çizim için not defterini hazırlayıp kullanacağımız işlevleri içe aktararak başlayalım:


In [ ]:
# imports_text.py
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib as mpl
plt.style.use('seaborn-whitegrid')
import numpy as np
import pandas as pd



## Örnek: Tatillerin ABD doğumlarına etkisi

Daha önce
    Örnek: Doğum oranı verisi bölümünde çalıştığımız veriye dönelim; takvim yılı boyunca ortalama doğumları çizmiştik.
    Aynı temizleme adımlarıyla başlayıp sonucu çizelim (aşağıdaki şekil):


In [ ]:
# shell command to download the data:
# !cd data && curl -O \
#   https://raw.githubusercontent.com/jakevdp/data-CDCbirths/master/births.csv



> **Not**
>


In [ ]:
# births_clean.py
from datetime import datetime

births = pd.read_csv('data/births.csv')

quartiles = np.percentile(births['births'], [25, 50, 75])
mu, sig = quartiles[1], 0.74 * (quartiles[2] - quartiles[0])
births = births.query('(births > @mu - 5 * @sig) & (births < @mu + 5 * @sig)')

births['day'] = births['day'].astype(int)

births.index = pd.to_datetime(10000 * births.year +
                              100 * births.month +
                              births.day, format='%Y%m%d')
births_by_date = births.pivot_table('births',
                                    [births.index.month, births.index.day])
births_by_date.index = [datetime(2012, month, day)
                        for (month, day) in births_by_date.index]



In [ ]:
# births_plot.py
fig, ax = plt.subplots(figsize=(12, 4))
births_by_date.plot(ax=ax);



Bu tür veriyi görselleştirirken okuyucunun dikkatini çekmek için çizimin belirli özelliklerini elle açıklamak yararlıdır.
  Bu, plt.text / ax.text ile belirli x/y konumlarına metin yerleştirerek yapılır (aşağıdaki şekil):


In [ ]:
# births_text_labels.py
fig, ax = plt.subplots(figsize=(12, 4))
births_by_date.plot(ax=ax)

# Add labels to the plot
style = dict(size=10, color='gray')

ax.text('2012-1-1', 3950, "New Year's Day", **style)
ax.text('2012-7-4', 4250, "Independence Day", ha='center', **style)
ax.text('2012-9-4', 4850, "Labor Day", ha='center', **style)
ax.text('2012-10-31', 4600, "Halloween", ha='right', **style)
ax.text('2012-11-25', 4450, "Thanksgiving", ha='center', **style)
ax.text('2012-12-25', 3850, "Christmas ", ha='right', **style)

# Label the axes
ax.set(title='USA births by day of year (1969-1988)',
       ylabel='average daily births')

# Format the x-axis with centered month labels
ax.xaxis.set_major_locator(mpl.dates.MonthLocator())
ax.xaxis.set_minor_locator(mpl.dates.MonthLocator(bymonthday=15))
ax.xaxis.set_major_formatter(plt.NullFormatter())
ax.xaxis.set_minor_formatter(mpl.dates.DateFormatter('%h'));



### 🧪 Şimdi deneyin

🧪 Türkçe tatil etiketi
      Grafiğe kendi dilinizde bir etiket ekleyin.
          
      ax.text('2012-5-1', 4000, 'İşçi Bayramı', ha='center', size=10, color='gray')

ax.text bir x konumu, y konumu, bir dize ve isteğe bağlı olarak renk, boyut, stil, hizalama gibi metin özelliklerini alır.
    Burada ha='right' ve ha='center' kullandık; ha horizontal alignment (yatay hizalama) kısaltmasıdır.
    Seçenekler için plt.text ve mpl.text.Text docstring'lerine bakın.

## Dönüşümler ve metin konumu

Önceki örnekte metni veri konumlarına sabitledik. Bazen metni veriden bağımsız olarak eksen veya figürün sabit bir konumuna sabitlemek iyidir.
    Matplotlib'de bu transform (dönüşüm) değiştirilerek yapılır.

Matplotlib birkaç koordinat sistemi kullanır: $(x, y) = (1, 1)$ veri noktası eksen veya figürde belirli bir konuma, o da ekranda belirli bir piksele karşılık gelir.
    Matematiksel olarak bu sistemler arası dönüşüm görece basittir; Matplotlib bunu matplotlib.transforms alt modülünde yapar.

Tipik kullanıcı dönüşüm ayrıntılarıyla nadiren uğraşır; metin yerleşiminde şu üç önceden tanımlı dönüşüm yararlıdır:

Bu dönüşümleri kullanarak çeşitli konumlara metin çizen bir örneğe bakalım (aşağıdaki şekil):


In [ ]:
# transforms_demo.py
fig, ax = plt.subplots(facecolor='lightgray')
ax.axis([0, 10, 0, 10])

# transform=ax.transData is the default, but we'll specify it anyway
ax.text(1, 5, ". Data: (1, 5)", transform=ax.transData)
ax.text(0.5, 0.1, ". Axes: (0.5, 0.1)", transform=ax.transAxes)
ax.text(0.2, 0.2, ". Figure: (0.2, 0.2)", transform=fig.transFigure);



### 🧪 Şimdi deneyin

🧪 transAxes ile başlık
      Eksenin üst ortasına figürden bağımsız bir not ekleyin.
          
      ax.text(0.5, 1.02, 'Üst not', transform=ax.transAxes, ha='center', va='bottom')

Matplotlib'in varsayılan metin hizalaması, her dizenin başındaki "." karakterinin belirtilen koordinata yaklaşık oturmasını sağlar.

transData x ve y eksen etiketleriyle ilişkili veri koordinatlarını verir.
    transAxes eksenin sol alt köşesinden (burada beyaz kutu) eksen boyutunun kesri olarak konum verir.
    transFigure benzerdir; konum figürün sol alt köşesinden (gri kutu) figür boyutunun kesri olarak verilir.

Eksen sınırlarını değiştirirsek yalnızca transData koordinatlarının etkilendiğini, diğerlerinin sabit kaldığını görün (aşağıdaki şekil):


In [ ]:
# transforms_limits.py
ax.set_xlim(0, 2)
ax.set_ylim(-6, 6)
fig



Bu davranış eksen sınırlarını etkileşimli değiştirerek daha net görülebilir: kodu not defterinde çalıştırıyorsanız %matplotlib inline yerine %matplotlib notebook kullanıp menüyle etkileşime geçebilirsiniz.

## Oklar ve annotate

İşaret çizgileri ve metinle birlikte yararlı bir başka açıklama basit oklardır.

plt.arrow vardır ancak önermem: oluşturduğu oklar SVG nesneleridir, en-boy oranı değişince hizalamak zorlaşır.
    Bunun yerine metin ve oku esnek tanımlayan plt.annotate önerilir.

İşte annotate'in birkaç seçeneğiyle gösterimi (aşağıdaki şekil):


In [ ]:
# annotate_demo.py
fig, ax = plt.subplots()

x = np.linspace(0, 20, 1000)
ax.plot(x, np.cos(x))
ax.axis('equal')

ax.annotate('local maximum', xy=(6.28, 1), xytext=(10, 4),
            arrowprops=dict(facecolor='black', shrink=0.05))

ax.annotate('local minimum', xy=(5 * np.pi, -1), xytext=(2, -6),
            arrowprops=dict(arrowstyle="->",
                            connectionstyle="angle3,angleA=0,angleB=-90"));



Ok stili arrowprops sözlüğüyle kontrol edilir; çok sayıda seçenek vardır.
    Matplotlib çevrimiçi belgelerinde iyi belgelenmiştir; burada birkaç örnekle gösterelim.
    Önceki doğum grafiğinde olası seçeneklerin bir kısmını kullanalım (aşağıdaki şekil):


In [ ]:
# births_annotate.py
fig, ax = plt.subplots(figsize=(12, 4))
births_by_date.plot(ax=ax)

# Add labels to the plot
ax.annotate("New Year's Day", xy=('2012-1-1', 4100),  xycoords='data',
            xytext=(50, -30), textcoords='offset points',
            arrowprops=dict(arrowstyle="->",
                            connectionstyle="arc3,rad=-0.2"))

ax.annotate("Independence Day", xy=('2012-7-4', 4250),  xycoords='data',
            bbox=dict(boxstyle="round", fc="none", ec="gray"),
            xytext=(10, -40), textcoords='offset points', ha='center',
            arrowprops=dict(arrowstyle="->"))

ax.annotate('Labor Day Weekend', xy=('2012-9-4', 4850), xycoords='data',
            ha='center', xytext=(0, -20), textcoords='offset points')
ax.annotate('', xy=('2012-9-1', 4850), xytext=('2012-9-7', 4850),
            xycoords='data', textcoords='data',
            arrowprops={'arrowstyle': '|-|,widthA=0.2,widthB=0.2', })

ax.annotate('Halloween', xy=('2012-10-31', 4600),  xycoords='data',
            xytext=(-80, -40), textcoords='offset points',
            arrowprops=dict(arrowstyle="fancy",
                            fc="0.6", ec="none",
                            connectionstyle="angle3,angleA=0,angleB=-90"))

ax.annotate('Thanksgiving', xy=('2012-11-25', 4500),  xycoords='data',
            xytext=(-120, -60), textcoords='offset points',
            bbox=dict(boxstyle="round4,pad=.5", fc="0.9"),
            arrowprops=dict(arrowstyle="->",
                            connectionstyle="angle,angleA=0,angleB=80,rad=20"))

ax.annotate('Christmas', xy=('2012-12-25', 3850),  xycoords='data',
             xytext=(-30, 0), textcoords='offset points',
             size=13, ha='right', va="center",
             bbox=dict(boxstyle="round", alpha=0.1),
             arrowprops=dict(arrowstyle="wedge,tail_width=0.5", alpha=0.1));

# Label the axes
ax.set(title='USA births by day of year (1969-1988)',
       ylabel='average daily births')

# Format the x-axis with centered month labels
ax.xaxis.set_major_locator(mpl.dates.MonthLocator())
ax.xaxis.set_minor_locator(mpl.dates.MonthLocator(bymonthday=15))
ax.xaxis.set_major_formatter(plt.NullFormatter())
ax.xaxis.set_minor_formatter(mpl.dates.DateFormatter('%h'));

ax.set_ylim(3600, 5400);



Seçenek çeşitliliği annotate'i güçlü ve esnek kılar; neredeyse istediğiniz ok stilini oluşturabilirsiniz.
    Ne yazık ki bu özellikler çoğu zaman elle ince ayar gerektirir; yayın kalitesi grafiklerde zaman alıcı olabilir!
    Son olarak, yukarıdaki stil karışımı veri sunumu için en iyi uygulama değildir; mevcut seçeneklerin gösterimi içindir.

Ok ve açıklama stilleri için Matplotlib
    Annotations öğreticisinde daha fazla tartışma ve örnek vardır.

> **Not**
>
